In [11]:
# train_contrastive.py
import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import sys, os

sys.path.append("..")  
from helpers.image_encoder import ImageEncoder
from helpers.text_encoder import TextEncoder
from helpers.loss import contrastive_loss 
from helpers.fusion import FusionModule
from helpers.projection import ProjectionModule
# from helpers.dataset import MultimodalDataset

from transformers import AutoTokenizer, AutoModel



In [ ]:
model_for_image = torch.hub.load('facebookresearch/dino:main', 'dino_vits16')
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
model_for_text = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")



Using cache found in /home/saad/.cache/torch/hub/facebookresearch_dino_main


ImportError: cannot import name 'trunc_normal_' from 'utils' (unknown location)

In [ ]:
# Define dataloader

In [ ]:

def train_one_epoch(dataloader, image_encoder, text_encoder, fusion_model,
                    icd_encoder, fused_proj, icd_proj, optimizer, device,
                    temperature=0.07):
    """
    Train for one epoch with contrastive loss, handling:
    - Multi-label ICDs per image
    - Missing image or text modalities
    """
    fusion_model.train()
    total_loss = 0

    for batch in dataloader:
        # batch is assumed to be a dict or tuple containing:
        # images, image_texts, icd_texts_list
        # icd_texts_list is a list of lists: each image/text may map to multiple ICDs
        images, image_texts, icd_texts_list = batch

        batch_size = len(images)
        x_img_list = []
        x_text_list = []

        # -----------------------
        # Encode image and text
        # -----------------------
        for i in range(batch_size):
            # Image embedding
            if images[i] is not None:
                x_img = image_encoder.encode(images[i]).to(device)
            else:
                x_img = torch.zeros(1, fusion_model.img_dim).to(device)
            x_img_list.append(x_img)

            # Text embedding
            if image_texts[i] is not None:
                x_text = text_encoder.encode(image_texts[i]).to(device)
            else:
                x_text = torch.zeros(1, fusion_model.text_dim).to(device)
            x_text_list.append(x_text)

        x_img_batch = torch.cat(x_img_list, dim=0)   # [B, img_dim]
        x_text_batch = torch.cat(x_text_list, dim=0) # [B, text_dim]

        # -----------------------
        # Fuse embeddings
        # -----------------------
        fusion_model = FusionModule().to(device)
        x_fused = fusion_model.fuse(x_img_batch, x_text_batch)

        # Project to common space (already normalized in fusion)
        x_fused_proj = fused_proj.project(x_fused)  # [B, D]
        x_fused_proj = F.normalize(x_fused_proj, dim=-1)

        # -----------------------
        # Prepare ICD embeddings
        # -----------------------
        # Flatten all ICDs in batch to compute similarity
        icd_flat_list = []
        icd_image_map = []  # map from icd index -> image index

        for img_idx, icd_texts in enumerate(icd_texts_list):
            for icd_text in icd_texts:
                x_icd = icd_encoder.encode(icd_text).to(device)
                icd_flat_list.append(x_icd)
                icd_image_map.append(img_idx)

        x_icd_proj = torch.cat(icd_flat_list, dim=0)  # [N_icd_total, D]
        x_icd_proj = icd_proj.project(x_icd_proj) # optional projection
        x_icd_proj = F.normalize(x_icd_proj, dim=-1)
        
        # -----------------------
        # Compute similarity
        # -----------------------
        # cosine similarity: [B, N_icd_total]
        sim_matrix = x_fused_proj @ x_icd_proj.T
        sim_matrix = sim_matrix / temperature

        # -----------------------
        # Compute multi-label contrastive loss
        # -----------------------
        # Create target mask: [B, N_icd_total] -> 1 if icd belongs to image
        target_mask = torch.zeros_like(sim_matrix)
        for icd_idx, img_idx in enumerate(icd_image_map):
            target_mask[img_idx, icd_idx] = 1.0

        # Use log-softmax and compute average loss over multiple positives
        log_prob = F.log_softmax(sim_matrix, dim=1)
        loss = -(target_mask * log_prob).sum(dim=1) / target_mask.sum(dim=1).clamp(min=1.0)
        loss = loss.mean()

        # -----------------------
        # Backpropagation
        # -----------------------
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


In [ ]:



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------
# Initialize modules
# ---------------------------

image_encoder_model = torch.hub.load('facebookresearch/dino:main', 'dino_vits16')  # pretrained DINO backbone
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
text_encoder_model = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")   # pretrained MedCLIP text encoder

image_encoder = ImageEncoder(image_encoder_model)
text_encoder = TextEncoder(text_encoder_model)

# Define fusion model
img_dim = 384          # DINO CLS
text_dim = 762         # MedCLIP
projected_dim = 512        # projected fused embedding
concatenated_dim = img_dim + text_dim

# Projection for fused image+text embeddings and ICD embeddings to common space
fused_proj = ProjectionModule(input_dim=concatenated_dim, projected_dim=projected_dim, method="linear").to(device)
icd_proj = ProjectionModule(input_dim=text_dim, projected_dim=projected_dim, method="linear").to(device)



# Optimizer
optimizer = optim.Adam(list(icd_proj.parameters()) + list(fused_proj.parameters()), 
    lr=1e-4
)

# ---------------------------
# Training loop
# ---------------------------

num_epochs = 10  # example

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    epoch_loss = train_one_epoch(
        dataloader=dataloader,
        image_encoder=image_encoder,
        text_encoder=text_encoder,
        icd_encoder=text_encoder,  # same text encoder for ICDs
        img_proj=fused_proj,
        icd_proj=icd_proj,
        optimizer=optimizer,
        device=device
    )
    print(f"Loss: {epoch_loss:.4f}")


NameError: name 'ImageEncoder' is not defined